# 03番(完全アニメ・yellow and orange idol dress)の「歩く」動画を作る — 全自動版
**使い方は、これだけです。**
1. 上のメニュー「ランタイム」→「ランタイムのタイプを変更」→ **T4 GPU** を選んで保存。
2. 上のメニュー「ランタイム」→ **「すべてのセルを実行」**。画像のアップロードは不要です(画像はこのノートブックに入っています)。
3. 数十分待つと、一番下で動画が再生され、自動でダウンロードされます(ダウンロードがブロックされたら、許可してください)。
4. ダウンロードされた mp4 を、チャットにアップロードしてください。

途中で赤いエラーが出たら、**その文字をそのまま**チャットに貼ってください。(未検証のため、止まる可能性があります)

In [ ]:
!pip -q install diffusers transformers accelerate sentencepiece imageio imageio-ffmpeg

In [ ]:
import base64, io
from PIL import Image, ImageFilter
IMG_B64 = "..."
img0 = Image.open(io.BytesIO(base64.b64decode(IMG_B64))).convert("RGB")
W, H = 720, 480
# 全身の縦長画像を、切らずに横長の中央に置く(左右は同じ画像をぼかして埋める)
bg = img0.resize((W, round(img0.height * W / img0.width)), Image.LANCZOS)
bg = bg.crop((0, (bg.height - H) // 2, W, (bg.height - H) // 2 + H)).filter(ImageFilter.GaussianBlur(25))
fg = img0.resize((round(img0.width * H / img0.height), H), Image.LANCZOS)
img = bg.copy()
img.paste(fg, ((W - fg.width) // 2, 0))
img

In [ ]:
import torch
from diffusers import CogVideoXImageToVideoPipeline
dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
pipe = CogVideoXImageToVideoPipeline.from_pretrained("THUDM/CogVideoX-5b-I2V", torch_dtype=dtype)
pipe.enable_sequential_cpu_offload()   # GPUメモリの節約(遅くなる)
pipe.vae.enable_tiling()
pipe.vae.enable_slicing()
print("読み込み完了。型:", dtype)

In [ ]:
from diffusers.utils import export_to_video
print("生成を始めます。数十分かかります(進み具合のバーが出ます)。")
prompt = ("A full-body anime girl with a short black bob and a yellow and orange idol dress walks forward confidently toward the camera, arms swinging naturally, her hair and skirt flowing, smooth walking motion, anime style, high quality")
frames = pipe(image=img, prompt=prompt, num_videos_per_prompt=1, num_inference_steps=30, num_frames=49,
              guidance_scale=6, generator=torch.Generator().manual_seed(1)).frames[0]
export_to_video(frames, "clip_cog.mp4", fps=8)
print("生成できました")

In [ ]:
!ffmpeg -y -loglevel error -i clip_cog.mp4 -vf "minterpolate=fps=24:mi_mode=mci" -pix_fmt yuv420p walk_03_B_clip.mp4
from IPython.display import Video, display
display(Video("walk_03_B_clip.mp4", embed=True, width=480))
from google.colab import files
files.download("walk_03_B_clip.mp4")